# Spatial-Temporal Attribution Analysis: Arousal Prediction

This notebook presents a systematic analysis of Layer-wise Relevance Propagation (LRP) attributions for models trained on the arousal prediction task. The analysis utilizes the `ExperimentResults` framework and `AttributionAnalyzer` to compute statistics across multiple experimental iterations.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import logging
import seaborn as sns
import matplotlib.pyplot as plt
import re

# Add src to path
sys.path.append(os.path.abspath("../src"))

from evaluate.analysis_utils import ExperimentResults, AttributionAnalyzer
from visualization.eeg_attribution import attribution_topomap, plot_attribution_summary
from evaluate.analyze import plot_summary_grid
from data.affect_prediction_datamodule import AffectPredictionDataModule
from utils import adapt_channel_names_to_mne, get_chan_names

sns.set_theme('paper', style='white', rc={
    'font.size': 10,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'figure.figsize': (5.5, 3)
})

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', stream=sys.stdout)
logger = logging.getLogger(__name__)

## 1. Experimental Result Discovery
Discovery and metadata extraction for all relevant experiment runs.

In [ ]:
EXPERIMENT_DIR_MAP = {
    'finetuned': '../../publish/arousal_finetuned',
    'from_scratch': '../../publish/arousal_from_scratch',
}

experiment_data = {}

for label, path in EXPERIMENT_DIR_MAP.items():
    if os.path.exists(path):
        results = ExperimentResults(path)
        analyzer = AttributionAnalyzer(results)
        experiment_data[label] = {'results': results, 'analyzer': analyzer}
        logger.info(f"Identified {label}: {len(results.run_dirs)} runs discovered.")
    else:
        logger.warning(f"Path not found: {path}")

## 2. Dataset Specification & Montage Definition
Loading the evaluation dataset and mapping channel indices to standard MNE nomenclature.

In [ ]:
DATA_DIR = "/path/to/your/data" 
datamodule = AffectPredictionDataModule(dataset_path="../data/brain_heart_emotion", 
                                        data_dir=DATA_DIR, 
                                        num_classes=2)
datamodule.setup()
test_ds = datamodule.test_ds
channel_names = adapt_channel_names_to_mne(get_chan_names(test_ds[0]["eeg_channel_indices"]))
logger.info(f"Dataset initialized. N={len(test_ds)} samples. Channels: {len(channel_names)}")

## 3. Statistical Aggregation of Neural Relevance
Computation of global and class-wise relevance statistics. Dask-based aggregation is employed to maintain numerical stability and manage memory constraints during the analysis of high-dimensional attribution tensors.

In [ ]:
for label, data in experiment_data.items():
    analyzer = data['analyzer']
    logger.info(f"Calculating stats for {label} (Run 0)...")
    ddf = analyzer.load_run_data(0, split='test')
    stats = analyzer.calculate_classwise_statistics(ddf, class_names={0: 'Low Arousal', 1: 'High Arousal'})

    # Aggregate mean across all classes (count-weighted), as in physionet_streamlined_analysis
    total_count = sum(cs['all_count'] for cs in stats.values())
    aggregate_mean = sum(cs['all_mean'] * cs['all_count'] for cs in stats.values()) / total_count
    plot_attribution_summary(
        aggregate_mean,
        channel_names=channel_names,
        title=f"{label} - All Classes (Aggregate)"
    )
    plt.show()

    for class_id, class_stats in stats.items():
        plot_attribution_summary(
            class_stats['correct_mean'],
            channel_names=channel_names,
            title=f"{label} - {class_stats['class_name']} (Correct Preds)"
        )
        plt.show()